# 01 Setup and Baseline

This notebook runs baseline (no finetuning) evaluation for `dataset1`, `dataset2`, and `dataset4` using `Qwen/Qwen2.5-1.5B-Instruct` by default.

Execution order:
1. Install dependencies
2. Configure model and quantization
3. Load model/tokenizer
4. Run benchmark on datasets 1/2/4
5. Save reports to `outputs/notebooks/baseline/`

In [1]:
# If needed (Colab), run once then restart runtime if prompted.
%pip install -q accelerate bitsandbytes datasets peft safetensors torch tqdm transformers

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 22.7 MB/s eta 0:00:00


In [2]:
# Compute check
import torch

print("Torch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU count:", torch.cuda.device_count())
    print("GPU name:", torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print("GPU memory (GB):", round(props.total_memory / (1024**3), 2))
else:
    print("Running on CPU runtime")

Torch version: 2.9.0+cu128
CUDA available: True
GPU count: 1
GPU name: NVIDIA A100-SXM4-40GB
GPU memory (GB): 39.49


In [3]:
from pathlib import Path
import json
import sys

from google.colab import drive
drive.mount('/content/drive')

ROOT = Path("/content/drive/MyDrive/abstention-data")

if not (ROOT / "data" / "dataset1.jsonl").exists() or not (ROOT / "src").exists():
    raise FileNotFoundError(
        f"Invalid ROOT: {ROOT}. Make sure repo is in Drive at /content/drive/MyDrive/abstention-data"
    )

SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("ROOT:", ROOT)

Mounted at /content/drive
ROOT: /content/drive/MyDrive/abstention-data


In [4]:
from abstention_pipeline.config import DEFAULT_MODEL_NAME, DEFAULT_SYSTEM_PROMPT, resolve_dataset_path
from abstention_pipeline.evaluation import run_benchmark, save_report
from abstention_pipeline.modeling import load_base_model, load_tokenizer

MODEL_NAME = DEFAULT_MODEL_NAME
QUANT_MODE = "4bit"  # none | 8bit | 4bit
SYSTEM_PROMPT = DEFAULT_SYSTEM_PROMPT
MAX_NEW_TOKENS = 32
BATCH_SIZE = 64
MAX_EXAMPLES = None
MAX_INPUT_TOKENS = 512

OUTPUT_DIR = ROOT / "outputs" / "notebooks" / "baseline"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Model:", MODEL_NAME)
print("Quantization:", QUANT_MODE)
print("Output:", OUTPUT_DIR)

Model: Qwen/Qwen2.5-1.5B-Instruct
Quantization: 4bit
Output: /content/drive/MyDrive/abstention-data/outputs/notebooks/baseline


In [5]:
tokenizer = load_tokenizer(MODEL_NAME)
model = load_base_model(MODEL_NAME, quantization_mode=QUANT_MODE, for_training=False)
model.eval()

model_max = getattr(tokenizer, "model_max_length", 4096)
if not isinstance(model_max, int) or model_max > 100000:
    model_max = 4096
MAX_INPUT_TOKENS = min(MAX_INPUT_TOKENS, model_max)

print("Model loaded.")
print("Using MAX_INPUT_TOKENS:", MAX_INPUT_TOKENS)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded.
Using MAX_INPUT_TOKENS: 512


In [6]:
baseline_summary = {
    "model_name": MODEL_NAME,
    "quantization": QUANT_MODE,
    "system_prompt": SYSTEM_PROMPT,
    "max_input_tokens": MAX_INPUT_TOKENS,
    "datasets": {},
}

for dataset_name in ["dataset1", "dataset2", "dataset4"]:
    ds_path = resolve_dataset_path(ROOT, dataset_name)
    report = run_benchmark(
        model=model,
        tokenizer=tokenizer,
        dataset_path=ds_path,
        system_prompt=SYSTEM_PROMPT,
        max_new_tokens=MAX_NEW_TOKENS,
        batch_size=BATCH_SIZE,
        max_examples=MAX_EXAMPLES,
        max_input_tokens=MAX_INPUT_TOKENS,
    )
    save_report(report, OUTPUT_DIR / f"{dataset_name}.json")
    baseline_summary["datasets"][dataset_name] = report["metrics"]
    print(f"\n{dataset_name} metrics:")
    print(json.dumps(report["metrics"], indent=2))

save_report(baseline_summary, OUTPUT_DIR / "summary.json")
print("\nSaved baseline summary:", OUTPUT_DIR / "summary.json")

Eval dataset1.jsonl:   0%|          | 0/157 [00:00<?, ?it/s]


dataset1 metrics:
{
  "n": 10000,
  "overall_exact_match": 0.0007,
  "answerable_exact_match": 0.0007,
  "abstain_precision": 0.0,
  "abstain_recall": 0.0,
  "abstain_f1": 0.0,
  "pred_abstain_rate": 0.0,
  "gold_abstain_rate": 0.0
}


Eval dataset2.jsonl:   0%|          | 0/157 [00:00<?, ?it/s]


dataset2 metrics:
{
  "n": 10000,
  "overall_exact_match": 0.0023,
  "answerable_exact_match": 0.001,
  "abstain_precision": 0.5,
  "abstain_recall": 0.0002,
  "abstain_f1": 0.0003998400639744103,
  "pred_abstain_rate": 0.0002,
  "gold_abstain_rate": 0.5
}


Eval dataset4.jsonl:   0%|          | 0/32 [00:00<?, ?it/s]


dataset4 metrics:
{
  "n": 2000,
  "overall_exact_match": 0.0005,
  "answerable_exact_match": 0.0,
  "abstain_precision": 1.0,
  "abstain_recall": 0.001,
  "abstain_f1": 0.0019980019980019984,
  "pred_abstain_rate": 0.0005,
  "gold_abstain_rate": 0.5
}

Saved baseline summary: /content/drive/MyDrive/abstention-data/outputs/notebooks/baseline/summary.json


In [7]:
# Optional quick view of saved summary
print((OUTPUT_DIR / "summary.json").read_text(encoding="utf-8")[:2000])

{
  "model_name": "Qwen/Qwen2.5-1.5B-Instruct",
  "quantization": "4bit",
  "system_prompt": "You are a careful question answering assistant. If the question cannot be answered reliably, output exactly <NO-ANSWER>. If it can be answered, output only the short answer with no extra words.",
  "max_input_tokens": 512,
  "datasets": {
    "dataset1": {
      "n": 10000,
      "overall_exact_match": 0.0007,
      "answerable_exact_match": 0.0007,
      "abstain_precision": 0.0,
      "abstain_recall": 0.0,
      "abstain_f1": 0.0,
      "pred_abstain_rate": 0.0,
      "gold_abstain_rate": 0.0
    },
    "dataset2": {
      "n": 10000,
      "overall_exact_match": 0.0023,
      "answerable_exact_match": 0.001,
      "abstain_precision": 0.5,
      "abstain_recall": 0.0002,
      "abstain_f1": 0.0003998400639744103,
      "pred_abstain_rate": 0.0002,
      "gold_abstain_rate": 0.5
    },
    "dataset4": {
      "n": 2000,
      "overall_exact_match": 0.0005,
      "answerable_exact_match": 0.